In [6]:
import polars as pl 
import duckdb 
import polars_ds as pds 
import requests 
import json 
from utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [11]:
from pathlib import Path

database_path = Path.cwd() / "ticket_sales.duckdb"

with duckdb.connect(str(database_path), read_only=True) as connection:
    snapshots = connection.sql("""
        SELECT
            CAST(snapshot_at AS VARCHAR) AS snapshot_at,
            event_id,
            section_id,
            section_name,
            total_seats,
            sold,
            available,
            unavailable,
            sold_out,
            other
        FROM ticket_snapshots
        ORDER BY snapshot_at, section_name
    """).pl()

snapshots


C:\Users\Yafee Ishraq\AppData\Local\Temp\ipykernel_16276\3277816103.py:20: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  """).pl()


snapshot_at,event_id,section_id,section_name,total_seats,sold,available,unavailable,sold_out,other
str,i32,i32,str,i32,i32,i32,i32,bool,i32
"""2026-08-24 10:10:41.580859+02""",1187151,755045,"""BT - Rullestol""",32,10,11,21,false,11
"""2026-08-24 10:10:41.580859+02""",1187151,755035,"""BT Felt A""",359,168,191,168,false,0
"""2026-08-24 10:10:41.580859+02""",1187151,755023,"""BT Felt A Øvre""",93,93,0,93,true,93
"""2026-08-24 10:10:41.580859+02""",1187151,755036,"""BT Felt B""",574,574,0,574,true,18
"""2026-08-24 10:10:41.580859+02""",1187151,755055,"""BT Felt C""",732,732,0,732,true,0
…,…,…,…,…,…,…,…,…,…
"""2026-08-24 10:11:59.456004+02""",1187151,755052,"""VIP 1""",147,34,113,34,false,0
"""2026-08-24 10:11:59.456004+02""",1187151,755063,"""VIP 2""",156,110,46,110,false,0
"""2026-08-24 10:11:59.456004+02""",1187151,755064,"""VIP 3""",156,89,63,93,false,4


In [2]:
available_events = get_available_events()

In [3]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [35]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [36]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [2]:
season_2026 = scrape_match_results(season_id=2025,year = 2026)

In [4]:
pl.DataFrame(season_2026).sort(by = 'date')

date,home_team,away_team,result,brann_table_position,snapshot_at,brann_goal_scorers
date,str,str,str,i64,"datetime[μs, UTC]",list[str]
2026-01-22,"""SK Brann""","""Midtjylland""","""3:3""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Holm"", ""E. Kornvig"", ""J. Soltvedt""]"
2026-01-29,"""Sturm Graz""","""SK Brann""","""1:0""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-02-19,"""SK Brann""","""Bologna""","""0:1""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-02-26,"""Bologna""","""SK Brann""","""1:0""",null,2026-08-23 08:53:52.539782 UTC,[]
2026-03-08,"""Tromsdalen""","""SK Brann""","""2:3 AET""",null,2026-08-23 08:53:52.539782 UTC,"[""F. Myhre"", ""J. Thorsteinsson"", ""J. Lungi Sørensen""]"
…,…,…,…,…,…,…
2026-07-12,"""SK Brann""","""IK Start""","""2:1""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Holm"", ""K. Eriksen""]"
2026-07-18,"""Molde FK""","""SK Brann""","""1:2""",null,2026-08-23 08:53:52.539782 UTC,"[""D. De Roeve"", ""K. Eriksen""]"
2026-07-26,"""SK Brann""","""Vålerenga""","""2:3""",null,2026-08-23 08:53:52.539782 UTC,"[""N. Castro"", ""F. Myhre""]"


In [4]:
paok = scrape_ticket_sections(1187151)

In [18]:
pl.DataFrame(paok).filter(pl.col('total_seats')==0,pl.col('sold')==0)

snapshot_at,event_id,section_id,section_name,total_seats,sold,available,unavailable,sold_out,other
"datetime[μs, UTC]",i64,i64,str,i64,i64,i64,i64,bool,i64
2026-08-22 20:20:11.449792 UTC,1187151,755042,"""FJORDKRAFT - STÅPLASSER""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755020,"""FJORDKRAFT Felt A""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755046,"""EGD Hjørnet""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755026,"""STORE STÅ NEDRE""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755021,"""FRYDENBØ Felt C Nedre""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755043,"""FRYDENBØ Felt B Nedre""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755038,"""STORE STÅ""",0,0,0,0,false,0
2026-08-22 20:20:11.449792 UTC,1187151,755041,"""Gangen Frydenbø""",0,0,0,0,false,0
